# EPOWER Energy Intelligence Demo

End-to-end setup of a governed, AI-powered data platform for a German energy retailer:

| Section | What it does |
|---------|-------------|
| **§1** Session Setup | Role, warehouse, Snowpark session |
| **§2** Database & Schema Setup | Database, schemas (Bronze/Silver/Gold/Ops) |
| **§3** Synthetic Domain Data | Sales, Billing, Service, HR, Finance — dim & fact tables (star schema) |
| **§4** Day-Ahead Electricity Prices | External access to **real** EPEX day-ahead prices (60-day backfill) |
| **§5** ePulse VPP Telemetry | Device registry + 60 days of **price-reactive** VPP telemetry |
| **§6** dbt Pipelines | Bronze → Silver → Gold transformation (VPP + Market Data) |
| **§7** Daily Task Scheduling | Automated daily refresh: prices → telemetry → dbt |
| **§8** Semantic Views | 7 Semantic Views for text-to-SQL via Cortex Analyst |
| **§9** Document Parsing & Cortex Search | PDF/MD parsing + 4 RAG search services |
| **§10** Intelligence Agent | Cortex Agent combining all capabilities |
| **§11** MCP Server | Model Context Protocol server for external AI clients |
| **§12** Verification | Object counts and data validation |

**Runtime**: ~15 minutes | **Prerequisites**: See README.md (Git API integration required)

---
## Architecture Overview

```
┌───────────────────────────────────────────────────────────────────┐
│                    SNOWFLAKE INTELLIGENCE AGENT                     │
└───────────────────────────────────────────────────────────────────┘
                                  │
         ┌────────────────────────┼────────────────────────┐
         ▼                        ▼                        ▼
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│ CORTEX ANALYST  │    │ CORTEX SEARCH   │    │   dbt MODELS    │
│  (Text-to-SQL)  │    │     (RAG)       │    │ (VPP Analytics) │
└─────────────────┘    └─────────────────┘    └─────────────────┘
         │                        │                        │
         ▼                        ▼                        ▼
┌───────────────────────────────────────────────────────────────────┐
│  EPOWER_GOLD (Dims/Facts/Views)   │  EPOWER_BRONZE → SILVER → GOLD    │
│  • customer_dim (20K)             │  • EPULSE_DEVICES (bridge)        │
│  • product_dim, sales_fact        │  • RAW_EPULSE_IOT_TELEMETRY       │
│  • billing_history, etc.          │  • dbt pipelines (VPP + Market)   │
└───────────────────────────────────────────────────────────────────┘
```

## 1. Session Setup

In [ ]:
# Import packages and get Snowpark session
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
-- Create role and warehouse
USE ROLE accountadmin;
CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS snowflake_intelligence_object_default;
CREATE OR REPLACE ROLE EPOWER_ROLE;
SET current_user_name = CURRENT_USER();
GRANT ROLE EPOWER_ROLE TO USER IDENTIFIER($current_user_name);
GRANT ROLE EPOWER_ROLE TO ROLE SYSADMIN;
GRANT CREATE DATABASE ON ACCOUNT TO ROLE EPOWER_ROLE;
GRANT EXECUTE TASK ON ACCOUNT TO ROLE EPOWER_ROLE;

CREATE OR REPLACE WAREHOUSE EPOWER_COMPUTE
    WAREHOUSE_SIZE = 'SMALL'
    GENERATION = '2'
    AUTO_SUSPEND = 300
    AUTO_RESUME = TRUE;
GRANT USAGE ON WAREHOUSE EPOWER_COMPUTE TO ROLE EPOWER_ROLE;

USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;

## 2. Database & Schema Setup

In [ ]:
%%sql
-- Core database and schemas (Medallion Architecture)
CREATE OR REPLACE DATABASE EPOWER_DEMO;
USE DATABASE EPOWER_DEMO;

CREATE SCHEMA IF NOT EXISTS EPOWER_BRONZE COMMENT = 'Raw data: IoT telemetry, API ingestion';
CREATE SCHEMA IF NOT EXISTS EPOWER_SILVER COMMENT = 'Cleaned & enriched data (dbt staging)';
CREATE SCHEMA IF NOT EXISTS EPOWER_GOLD   COMMENT = 'Business-ready: dimensions, facts, metrics (dbt marts)';
CREATE SCHEMA IF NOT EXISTS EPOWER_OPS    COMMENT = 'Pipeline operations: stored procs, tasks, stages, network rules';

-- File format for CSV loading
CREATE OR REPLACE FILE FORMAT EPOWER_DEMO.EPOWER_OPS.CSV_FORMAT
    TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"' TRIM_SPACE = TRUE
    ERROR_ON_COLUMN_COUNT_MISMATCH = FALSE NULL_IF = ('NULL', 'null', '', 'N/A');

## 3. Synthetic Domain Data (Sales, Products, HR, Finance)

Loads EPOWER's core business data — **synthetic** data representing a German energy retailer with 20,000 customers across 6 business domains: **Sales & Contracts**, **Billing & Consumption**, **Customer Service**, **HR & Workforce**, **Finance**, and **Marketing**. The data is organized as a star schema with **13 dimension tables** (customers, products, vendors, regions, employees, etc.) and **10 fact tables** (sales, billing, service tickets, HR records, finance transactions, marketing campaigns, CRM data).

All CSV files are uploaded from the workspace to an internal stage, then loaded into `EPOWER_GOLD`.

In [ ]:
%%sql
-- Internal stage for data and documents
USE ROLE EPOWER_ROLE;
USE SCHEMA EPOWER_DEMO.EPOWER_OPS;

-- Create internal stage with directory table
CREATE OR REPLACE STAGE EPOWER_STAGE FILE_FORMAT = CSV_FORMAT DIRECTORY = (ENABLE = TRUE);

In [ ]:
# Upload workspace files to internal stage
import os

stage_name = '@EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE'
base_path = '../demo_data'

for folder in ['structured_data', 'unstructured_data']:
    local_path = f'{base_path}/{folder}'
    for root, dirs, files in os.walk(local_path):
        for file in files:
            file_path = os.path.join(root, file)
            rel_path = os.path.relpath(root, base_path)
            stage_path = f'{stage_name}/{rel_path}/'
            session.file.put(file_path, stage_path, auto_compress=False, overwrite=True)
            print(f'  ✓ {rel_path}/{file}')

print('\n✓ Files uploaded to stage')
session.sql('ALTER STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE REFRESH').collect()

### Dimension Tables — Customers, Products, Vendors, Regions, Employees

13 dimension tables providing the WHO, WHAT, WHERE, WHEN context across all business domains:

| Domain | Tables | Key Data |
|--------|--------|----------|
| **Customers & Sales** | `customer_dim`, `product_dim`, `product_category_dim`, `vendor_dim`, `sales_rep_dim`, `region_dim` | 20K customers, 15 products across 6 energy categories (with CAPEX/OPEX pricing), sales regions |
| **HR & Workforce** | `employee_dim`, `job_dim`, `department_dim`, `location_dim` | 1,000 employees with job levels and locations |
| **Marketing** | `campaign_dim`, `channel_dim` | Campaign definitions and marketing channels |
| **Finance** | `account_dim` | Financial account structure |

In [ ]:
%%sql
-- All dimension tables in EPOWER_GOLD
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.product_category_dim (category_key INT PRIMARY KEY, category_name VARCHAR(100) NOT NULL, vertical VARCHAR(50) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.product_dim (product_key INT PRIMARY KEY, product_name VARCHAR(200) NOT NULL, category_key INT NOT NULL, category_name VARCHAR(100), vertical VARCHAR(50), capex_eur DECIMAL(12,2) DEFAULT 0, opex_eur_year DECIMAL(10,2) DEFAULT 0);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.customer_dim (customer_key INT PRIMARY KEY, customer_name VARCHAR(200) NOT NULL, customer_type VARCHAR(50), housing_type VARCHAR(100), vertical VARCHAR(50), address VARCHAR(200), city VARCHAR(100), state VARCHAR(50), zip VARCHAR(20), region_key INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.vendor_dim (vendor_key INT PRIMARY KEY, vendor_name VARCHAR(200) NOT NULL, vendor_type VARCHAR(100), vertical VARCHAR(50), address VARCHAR(200), city VARCHAR(100), state VARCHAR(50), zip VARCHAR(20));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.account_dim (account_key INT PRIMARY KEY, account_name VARCHAR(100) NOT NULL, account_type VARCHAR(50));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.department_dim (department_key INT PRIMARY KEY, department_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.region_dim (region_key INT PRIMARY KEY, region_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sales_rep_dim (sales_rep_key INT PRIMARY KEY, rep_name VARCHAR(200) NOT NULL, hire_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.campaign_dim (campaign_key INT PRIMARY KEY, campaign_name VARCHAR(300) NOT NULL, objective VARCHAR(100));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.channel_dim (channel_key INT PRIMARY KEY, channel_name VARCHAR(100) NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.employee_dim (employee_key INT PRIMARY KEY, employee_name VARCHAR(200) NOT NULL, gender VARCHAR(1), hire_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.job_dim (job_key INT PRIMARY KEY, job_title VARCHAR(100) NOT NULL, job_level INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.location_dim (location_key INT PRIMARY KEY, location_name VARCHAR(200) NOT NULL);

### Fact Tables — Sales, Billing, Service, HR, Finance, Marketing & CRM

10 fact/transactional tables capturing business activity across all domains:

| Domain | Tables | Volume |
|--------|--------|--------|
| **Sales & Contracts** | `sales_fact` | 260K contract records with revenue, units, dates |
| **Billing & Consumption** | `billing_history`, `customer_products` | 540K billing records, product ownership |
| **Customer Service** | `service_logs` | 10K tickets with topic, sentiment, priority |
| **HR & Workforce** | `hr_employee_fact` | 12K records with salary, department, attrition |
| **Finance** | `finance_transactions` | 30K transactions with approvals |
| **Marketing** | `marketing_campaign_fact` | 141K campaign metrics (spend, leads, impressions) |
| **CRM (Salesforce)** | `sf_accounts`, `sf_opportunities`, `sf_contacts` | Simulated Salesforce CRM data linked to sales |

In [ ]:
%%sql
-- Core fact tables in EPOWER_GOLD
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sales_fact (sale_id INT PRIMARY KEY, date DATE NOT NULL, customer_key INT NOT NULL, product_key INT NOT NULL, sales_rep_key INT NOT NULL, region_key INT NOT NULL, vendor_key INT NOT NULL, amount DECIMAL(12,2) NOT NULL, units INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.billing_history (billing_id INT PRIMARY KEY, customer_key INT NOT NULL, billing_date DATE NOT NULL, billing_type VARCHAR(50) NOT NULL, consumption_kwh INT NOT NULL, amount DECIMAL(10,2) NOT NULL, payment_status VARCHAR(50));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.service_logs (log_id INT PRIMARY KEY, customer_key INT NOT NULL, log_date DATE NOT NULL, topic VARCHAR(100) NOT NULL, category VARCHAR(100), description VARCHAR(500), sentiment VARCHAR(50), channel VARCHAR(50), priority VARCHAR(50), resolution_date DATE, agent_key INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.finance_transactions (transaction_id INT PRIMARY KEY, date DATE NOT NULL, account_key INT NOT NULL, department_key INT NOT NULL, vendor_key INT NOT NULL, product_key INT NOT NULL, customer_key INT NOT NULL, amount DECIMAL(12,2) NOT NULL, approval_status VARCHAR(20), procurement_method VARCHAR(50), approver_id INT, approval_date DATE, purchase_order_number VARCHAR(50), contract_reference VARCHAR(100));
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.marketing_campaign_fact (campaign_fact_id INT PRIMARY KEY, date DATE NOT NULL, campaign_key INT NOT NULL, product_key INT NOT NULL, channel_key INT NOT NULL, region_key INT NOT NULL, spend DECIMAL(10,2) NOT NULL, leads_generated INT NOT NULL, impressions INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.hr_employee_fact (hr_fact_id INT PRIMARY KEY, date DATE NOT NULL, employee_key INT NOT NULL, department_key INT NOT NULL, job_key INT NOT NULL, location_key INT NOT NULL, salary DECIMAL(10,2) NOT NULL, attrition_flag INT NOT NULL);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.customer_products (customer_product_id INT PRIMARY KEY, customer_key INT NOT NULL, product_key INT NOT NULL, category_key INT NOT NULL, category_name VARCHAR(100) NOT NULL, acquisition_date DATE NOT NULL, status VARCHAR(20) DEFAULT 'Active');

-- Salesforce CRM tables
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_accounts (account_id VARCHAR(20) PRIMARY KEY, account_name VARCHAR(200) NOT NULL, customer_key INT NOT NULL, industry VARCHAR(100), vertical VARCHAR(50), billing_street VARCHAR(200), billing_city VARCHAR(100), billing_state VARCHAR(50), billing_postal_code VARCHAR(20), account_type VARCHAR(50), annual_revenue DECIMAL(15,2), employees INT, created_date DATE);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_opportunities (opportunity_id VARCHAR(20) PRIMARY KEY, sale_id INT, account_id VARCHAR(20) NOT NULL, opportunity_name VARCHAR(200) NOT NULL, stage_name VARCHAR(100) NOT NULL, amount DECIMAL(15,2) NOT NULL, probability DECIMAL(5,2), close_date DATE, created_date DATE, lead_source VARCHAR(100), type VARCHAR(100), campaign_id INT);
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.sf_contacts (contact_id VARCHAR(20) PRIMARY KEY, opportunity_id VARCHAR(20) NOT NULL, account_id VARCHAR(20) NOT NULL, first_name VARCHAR(100), last_name VARCHAR(100), email VARCHAR(200), phone VARCHAR(50), title VARCHAR(100), department VARCHAR(100), lead_source VARCHAR(100), campaign_no INT, created_date DATE);

###  Data Ingestion — Load 23 Tables into EPOWER_GOLD

Loads all dimension and fact tables from the internal stage (`@EPOWER_STAGE/structured_data/`) into `EPOWER_GOLD`. Each CSV file maps 1:1 to a table created above. Total: **~1M+ rows** across 23 tables covering all 6 business domains.

In [ ]:
# Load all tables from stage into EPOWER_GOLD
tables = [
    "product_category_dim", "product_dim", "customer_dim", "vendor_dim", "account_dim",
    "department_dim", "region_dim", "sales_rep_dim", "campaign_dim", "channel_dim",
    "employee_dim", "job_dim", "location_dim", "sales_fact", "billing_history",
    "service_logs", "finance_transactions", "marketing_campaign_fact", "hr_employee_fact",
    "customer_products", "sf_accounts", "sf_opportunities", "sf_contacts"
]

print("Loading tables from stage into EPOWER_GOLD...")
for t in tables:
    try:
        r = session.sql(f"COPY INTO EPOWER_DEMO.EPOWER_GOLD.{t} FROM @EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/structured_data/{t}.csv FILE_FORMAT=EPOWER_DEMO.EPOWER_OPS.CSV_FORMAT ON_ERROR='CONTINUE'").collect()
        print(f"  \u2713 {t}: {r[0]['rows_loaded']} rows")
    except: print(f"  \u26a0 {t}: skipped")
print("\n\u2713 Data loading complete")

In [ ]:
%%sql
-- Verify core data
SELECT 'customer_dim' AS table_name, COUNT(*) AS row_count FROM EPOWER_DEMO.EPOWER_GOLD.customer_dim
UNION ALL SELECT 'product_dim', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.product_dim
UNION ALL SELECT 'sales_fact', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.sales_fact
UNION ALL SELECT 'customer_products', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.customer_products;

## 4. Day-Ahead Electricity Prices (Real Market Data)

Sets up external access to the [Energy-Charts API](https://api.energy-charts.info/) by Fraunhofer ISE and ingests **real day-ahead electricity prices** for the DE-LU (Germany/Luxembourg) bidding zone from the European energy exchange (EPEX Spot).

Day-ahead prices are published daily around **13:00 CET** and determine wholesale electricity costs for the next 24 hours in **15-minute intervals** (96 entries per delivery day).

**Why prices come first:** The VPP telemetry generated in Section 5 uses real prices to simulate realistic battery charge/discharge behavior. Loading prices first means every telemetry record correlates with actual market data — no heuristics needed.

| Step | What |
|------|------|
| **4a** | Create `RAW_DAY_AHEAD_PRICES` table + API network access |
| **4b** | Define `FETCH_DAY_AHEAD_PRICES(TARGET_DATE)` — single-day fetch |
| **4c** | Define `BACKFILL_DAY_AHEAD_PRICES()` — 60-day range fetch (single API call, CET delivery-day aligned) |
| **4d** | Execute backfill + fetch today's prices |

```
Energy-Charts API ──► RAW_DAY_AHEAD_PRICES (Bronze) ──► dbt (Section 6) ──► Silver/Gold
                          │
                     1 row per delivery day
                     96 × 15-min price entries
                     CET-aligned (22:00 day-1 to 21:45 day)
```

In [ ]:
%%sql
USE SCHEMA EPOWER_BRONZE;

CREATE OR REPLACE TABLE RAW_DAY_AHEAD_PRICES (fetch_timestamp TIMESTAMP_LTZ DEFAULT CURRENT_TIMESTAMP(), fetch_date DATE, raw_data VARIANT);

CREATE OR REPLACE NETWORK RULE EPOWER_DEMO.EPOWER_OPS.ENERGY_CHARTS_NETWORK_RULE
    MODE = EGRESS TYPE = HOST_PORT VALUE_LIST = ('api.energy-charts.info:443');

USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION energy_charts_integration
    ALLOWED_NETWORK_RULES = (EPOWER_DEMO.EPOWER_OPS.ENERGY_CHARTS_NETWORK_RULE)
    ENABLED = true;
GRANT USAGE ON INTEGRATION energy_charts_integration TO ROLE EPOWER_ROLE;
USE ROLE EPOWER_ROLE;

In [ ]:
%%sql
CREATE OR REPLACE PROCEDURE EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(TARGET_DATE DATE)
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
HANDLER = 'fetch_prices'
EXTERNAL_ACCESS_INTEGRATIONS = (energy_charts_integration)
AS
$$
import requests
import json
from datetime import date, timedelta

def fetch_prices(session, target_date):
    td = target_date if isinstance(target_date, date) else date.fromisoformat(str(target_date)[:10])

    existing = session.sql(f"SELECT COUNT(*) as cnt FROM EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES WHERE fetch_date = '{td}'").collect()
    if existing[0]['CNT'] > 0:
        return f"Skipped: Data for {td} already exists"

    url = "https://api.energy-charts.info/price"
    params = {"bzn": "DE-LU", "start": td.isoformat()}

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        json_str = json.dumps(data).replace("'", "''")
        session.sql(
            f"INSERT INTO EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES (fetch_date, raw_data) "
            f"SELECT '{td}', PARSE_JSON('{json_str}')"
        ).collect()

        record_count = len(data.get('unix_seconds', []))
        return f"Success: Fetched {record_count} price records for {td}"

    except requests.exceptions.RequestException as e:
        return f"Error fetching data: {str(e)}"
    except Exception as e:
        return f"Error: {str(e)}"
$$;

In [ ]:
%%sql
-- Backfill last 60 days of day-ahead prices using a single API range call
-- Splits the response into one row per delivery day (96 entries = 15-min intervals)
-- Delivery day alignment: CET 22:00 (day-1) to CET 21:45 (day) per EPEX convention
-- Idempotent: skips dates that already exist in RAW_DAY_AHEAD_PRICES
CREATE OR REPLACE PROCEDURE EPOWER_OPS.BACKFILL_DAY_AHEAD_PRICES()
RETURNS STRING
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
HANDLER = 'backfill_prices'
EXTERNAL_ACCESS_INTEGRATIONS = (energy_charts_integration)
AS
$$
import requests
import json
from datetime import date, timedelta, datetime, timezone

def backfill_prices(session):
    today = date.today()
    start_date = today - timedelta(days=59)

    existing_rows = session.sql(
        f"SELECT DISTINCT fetch_date FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES "
        f"WHERE fetch_date BETWEEN '{start_date}' AND '{today}'"
    ).collect()
    existing_dates = {row['FETCH_DATE'] for row in existing_rows}

    all_dates = {start_date + timedelta(days=i) for i in range(60)}
    missing_dates = sorted(all_dates - existing_dates)

    if not missing_dates:
        return f"Backfill skipped: all {len(all_dates)} days already exist ({start_date} to {today})"

    url = "https://api.energy-charts.info/price"
    params = {"bzn": "DE-LU", "start": missing_dates[0].isoformat(), "end": missing_dates[-1].isoformat()}

    try:
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return f"Error fetching data: {str(e)}"

    unix_seconds = data.get('unix_seconds', [])
    prices = data.get('price', [])
    if not unix_seconds:
        return "Error: API returned no data"

    from zoneinfo import ZoneInfo
    cet = ZoneInfo('Europe/Berlin')

    day_buckets = {}
    for i, ts in enumerate(unix_seconds):
        dt_cet = datetime.fromtimestamp(ts, tz=timezone.utc).astimezone(cet)
        delivery_day = dt_cet.date() + timedelta(days=1) if dt_cet.hour >= 22 else dt_cet.date()
        if delivery_day not in day_buckets:
            day_buckets[delivery_day] = {'unix_seconds': [], 'price': []}
        day_buckets[delivery_day]['unix_seconds'].append(ts)
        day_buckets[delivery_day]['price'].append(prices[i] if i < len(prices) else None)

    inserted = 0
    skipped = 0
    errors = []

    for day_date in sorted(day_buckets.keys()):
        if day_date in existing_dates:
            skipped += 1
            continue

        day_data = {
            'license_info': data.get('license_info', ''),
            'unix_seconds': day_buckets[day_date]['unix_seconds'],
            'price': day_buckets[day_date]['price'],
            'unit': data.get('unit', 'EUR/MWh'),
            'deprecated': data.get('deprecated', False)
        }

        try:
            json_str = json.dumps(day_data).replace("'", "''")
            session.sql(
                f"INSERT INTO EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES (fetch_date, raw_data) "
                f"SELECT '{day_date}', PARSE_JSON('{json_str}')"
            ).collect()
            inserted += 1
        except Exception as e:
            errors.append(f"{day_date}: {str(e)}")

    result = f"Backfill complete: {inserted} days inserted, {skipped} skipped ({start_date} to {today})"
    if errors:
        result += f", {len(errors)} errors: " + "; ".join(errors[:3])
    return result
$$;

In [ ]:
%%sql
-- Backfill 60 days of day-ahead prices (single API range call, idempotent)
CALL EPOWER_DEMO.EPOWER_OPS.BACKFILL_DAY_AHEAD_PRICES();

-- Fetch today's prices (in case backfill missed the current day)
CALL EPOWER_DEMO.EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(CURRENT_DATE());

-- Verify
SELECT COUNT(*) AS total_days,
       MIN(fetch_date) AS earliest_date,
       MAX(fetch_date) AS latest_date
FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES;

In [ ]:
%%sql -r dataframe_4
select * from EPOWER_DEMO.EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES;

## 5. ePulse VPP Telemetry Data

EPOWER operates an **ePulse Virtual Power Plant (VPP)** — a network of ~4,050 residential battery storage systems that are remotely coordinated to stabilize the electricity grid and optimize energy trading. These are customers who purchased the **Solar L 12kWp + Speicher 10kWh** package (the only product that includes battery storage). About 90% of the ~4,500 battery owners are enrolled in the VPP program and have an **ePulse gateway** that reports hourly telemetry data.

### Price-Reactive Battery Strategy

The VPP simulates real battery behavior: batteries **charge when electricity is cheap** and **discharge when expensive** — modeled after real VPP aggregators like 1KOMMA5° Heartbeat AI. Since we loaded real day-ahead prices in Section 4, every telemetry record correlates with actual market data:

| Price Zone | Battery Action | Grid Flow | SOC Target |
|-----------|---------------|-----------|------------|
| **Negative** (< €0/MWh) | Max Charge | Import +3.5 to +5 kW | 85–95% |
| **Low** (< P25) | Charge | Import +2 to +4.5 kW | 70–92% |
| **Medium** (P25–P75) | Self-Consume | Small ±1.5 kW | 40–70% |
| **High** (> P75) | Discharge | Export -2 to -5 kW | 12–35% |

### Data Model

| Table | Description |
|-------|-------------|
| `EPULSE_DEVICES` | Device registry linking customers to IoT gateways (solar, battery, heat pump capabilities) |
| `RAW_EPULSE_IOT_TELEMETRY` | Hourly readings: solar yield (kW), battery SOC (%), heat pump consumption (kW), grid import/export (kW) |

```
customer_dim (EPOWER_GOLD) ◄──customer_key──► EPULSE_DEVICES (EPOWER_BRONZE)
                                              │
                              RAW_DAY_AHEAD_PRICES ──► price-reactive logic
                                              │
                                              ▼
                                  RAW_EPULSE_IOT_TELEMETRY (EPOWER_BRONZE)
```

The telemetry covers the **same 60-day window** as the day-ahead prices, ensuring full price correlation across the entire dataset.

### Step 5a: Device Registry (EPULSE_DEVICES)

The device registry is **derived** from the customer product data loaded in Section 3 — it is not a separate CSV import. The SQL below queries `customer_products` joined with `product_dim` to determine which customers own solar panels, battery storage, or heat pumps. Key Snowflake SQL features used here:

- **CTE (Common Table Expression)** — the `WITH hw AS (...)` clause computes hardware flags per customer before the final `INSERT`
- **Conditional Aggregation** — `MAX(CASE WHEN ... THEN 1 ELSE 0 END) = 1` pivots product categories into boolean flags (`has_solar`, `has_battery`, `has_heatpump`)
- **Deterministic Hashing** — `MOD(ABS(HASH(customer_key)), 100)` generates a repeatable pseudo-random enrollment rate (~90% of battery owners enroll in VPP), ensuring consistent results across runs

Only customers who own at least one hardware product (solar, battery, or heat pump) get a device record — pure electricity/gas tariff customers are excluded.

In [ ]:
%%sql
USE SCHEMA EPOWER_BRONZE;

CREATE OR REPLACE TABLE EPULSE_DEVICES (
    customer_key INT, gateway_id VARCHAR(20), has_solar BOOLEAN, has_battery BOOLEAN,
    has_heatpump BOOLEAN, is_vpp_enrolled BOOLEAN, enrollment_date DATE
)
COMMENT = 'Device registry linking customers to ePulse IoT gateways. Each record represents a device with solar, battery, and heat pump capabilities plus VPP enrollment status.';

CREATE OR REPLACE TABLE RAW_EPULSE_IOT_TELEMETRY (
    ts TIMESTAMP_NTZ, gateway_id VARCHAR(20), customer_key INT,
    solar_yield_kw FLOAT, battery_soc_pct FLOAT, heatpump_consumption_kw FLOAT, grid_import_export_kw FLOAT
);

INSERT INTO EPULSE_DEVICES
WITH hw AS (
    SELECT cp.customer_key,
        MAX(CASE WHEN cp.category_name='Solar & Storage' THEN 1 ELSE 0 END)=1 AS has_solar,
        MAX(CASE WHEN p.product_name LIKE '%Speicher%' THEN 1 ELSE 0 END)=1 AS has_battery,
        MAX(CASE WHEN cp.category_name='Heat Pumps' THEN 1 ELSE 0 END)=1 AS has_heatpump,
        MIN(cp.acquisition_date) AS first_acq
    FROM EPOWER_DEMO.EPOWER_GOLD.customer_products cp
    LEFT JOIN EPOWER_DEMO.EPOWER_GOLD.product_dim p ON cp.product_key=p.product_key
    WHERE cp.status='Active' GROUP BY cp.customer_key
)
SELECT customer_key, 'GW-'||LPAD(customer_key::VARCHAR,5,'0'),
    has_solar, has_battery, has_heatpump,
    CASE WHEN has_battery AND MOD(ABS(HASH(customer_key)),100)<90 THEN TRUE ELSE FALSE END,
    CASE WHEN has_battery AND MOD(ABS(HASH(customer_key)),100)<90 THEN DATEADD('day',30,first_acq) END
FROM hw WHERE has_solar OR has_battery OR has_heatpump;

SELECT COUNT(*) AS devices, SUM(CASE WHEN is_vpp_enrolled THEN 1 ELSE 0 END) AS vpp_enrolled FROM EPULSE_DEVICES;

### Step 5b: Telemetry Generation Procedure

The cell below creates a **Snowflake Stored Procedure** (`GENERATE_DAILY_TELEMETRY`) that populates hourly IoT telemetry for all battery-equipped devices. This procedure is designed to be **idempotent** — it checks the latest timestamp in the telemetry table and only generates data for missing days, making it safe to call repeatedly or from a scheduled task.

**Snowflake features leveraged:**

- **SQL Stored Procedures** — Snowflake supports procedural logic (variables, IF/THEN, loops) directly in SQL via the `EXECUTE AS OWNER` model. No external language runtime needed.
- **Generator Function** — `TABLE(GENERATOR(ROWCOUNT => n))` produces synthetic row sets on the fly without requiring a physical table. Combined with `seq4()`, it creates a sequence of hourly timestamps.
- **CROSS JOIN for Cartesian Products** — Each device is crossed with each hour to produce one telemetry record per device per hour (~4,500 battery devices × 24 hours × 60 days ≈ **6.5 million records**).
- **Semi-Structured Data Access** — The procedure reads EPEX spot prices from the `RAW_DAY_AHEAD_PRICES` table using Snowflake's **variant column** notation (`raw_data:price[index]::FLOAT`), parsing JSON arrays directly in SQL.
- **LATERAL FLATTEN** — Expands JSON arrays into relational rows for joining with the hourly time series.
- **Price-Reactive Logic** — Battery SOC and grid flow values are computed using `APPROX_PERCENTILE` quartile boundaries from real market prices, simulating realistic VPP dispatch behavior.

In [ ]:
%%sql
-- Generate 60 days of price-reactive VPP telemetry
-- Uses real day-ahead prices from RAW_DAY_AHEAD_PRICES (loaded in Section 4)
-- Battery SOC and grid flow correlate with actual EPEX spot prices
-- Covers the same 60-day window as the price data

CREATE OR REPLACE PROCEDURE EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY()
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS OWNER
AS
$$
BEGIN
    LET days_generated INT := 0;
    LET max_ts TIMESTAMP_NTZ;
    LET start_date DATE;
    LET end_date DATE := CURRENT_DATE();

    SELECT MAX(ts)::DATE INTO :max_ts FROM EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY;

    IF (:max_ts IS NULL) THEN
        start_date := DATEADD(DAY, -59, :end_date);
    ELSE
        start_date := DATEADD(DAY, 1, :max_ts::DATE);
    END IF;

    IF (:start_date > :end_date) THEN
        RETURN 'Skipped: Telemetry already up to date (latest: ' || :max_ts::VARCHAR || ')';
    END IF;

    LET num_days INT := DATEDIFF(DAY, :start_date, :end_date) + 1;
    LET num_hours INT := :num_days * 24;

    INSERT INTO EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY
    WITH hours AS (
        SELECT DATEADD(HOUR, seq4(), :start_date::TIMESTAMP_NTZ) AS ts
        FROM TABLE(GENERATOR(ROWCOUNT => :num_hours))
    ),
    devices AS (
        SELECT customer_key, gateway_id, has_solar, has_heatpump, is_vpp_enrolled
        FROM EPOWER_BRONZE.EPULSE_DEVICES WHERE has_battery
    ),
    price_stats AS (
        SELECT
            APPROX_PERCENTILE(r.raw_data:price[ts.INDEX]::FLOAT, 0.25) AS p25,
            APPROX_PERCENTILE(r.raw_data:price[ts.INDEX]::FLOAT, 0.75) AS p75
        FROM EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES r,
        LATERAL FLATTEN(input => r.raw_data:unix_seconds) ts
    ),
    hourly_prices AS (
        SELECT DATE_TRUNC('HOUR', TO_TIMESTAMP_NTZ(ts.VALUE::NUMBER)) AS price_hour,
               AVG(r.raw_data:price[ts.INDEX]::FLOAT) AS price_eur_mwh
        FROM EPOWER_BRONZE.RAW_DAY_AHEAD_PRICES r,
        LATERAL FLATTEN(input => r.raw_data:unix_seconds) ts
        GROUP BY 1
    )
    SELECT
        h.ts,
        d.gateway_id,
        d.customer_key,
        CASE WHEN d.has_solar AND HOUR(h.ts) BETWEEN 6 AND 20
             THEN ROUND(GREATEST(0, UNIFORM(0.5,8.0,RANDOM()) * (1 - ABS(HOUR(h.ts)-13)*0.08)), 2)
             ELSE 0 END AS solar_yield_kw,
        CASE
            WHEN NOT d.is_vpp_enrolled THEN ROUND(UNIFORM(20.0, 80.0, RANDOM()), 1)
            WHEN p.price_eur_mwh IS NULL THEN ROUND(UNIFORM(30.0, 70.0, RANDOM()), 1)
            WHEN p.price_eur_mwh < 0           THEN ROUND(UNIFORM(85.0, 95.0, RANDOM()), 1)
            WHEN p.price_eur_mwh < ps.p25      THEN ROUND(UNIFORM(70.0, 92.0, RANDOM()), 1)
            WHEN p.price_eur_mwh > ps.p75      THEN ROUND(UNIFORM(12.0, 35.0, RANDOM()), 1)
            ELSE ROUND(UNIFORM(40.0, 70.0, RANDOM()), 1)
        END AS battery_soc_pct,
        CASE WHEN d.has_heatpump THEN ROUND(UNIFORM(0.5, 4.5, RANDOM()), 2) ELSE 0 END AS heatpump_consumption_kw,
        CASE
            WHEN NOT d.is_vpp_enrolled THEN ROUND(UNIFORM(-2.0, 5.0, RANDOM()), 2)
            WHEN p.price_eur_mwh IS NULL THEN ROUND(UNIFORM(-1.0, 3.0, RANDOM()), 2)
            WHEN p.price_eur_mwh < 0           THEN ROUND(UNIFORM(3.5, 5.0, RANDOM()), 2)
            WHEN p.price_eur_mwh < ps.p25      THEN ROUND(UNIFORM(2.0, 4.5, RANDOM()), 2)
            WHEN p.price_eur_mwh > ps.p75      THEN ROUND(UNIFORM(-5.0, -2.0, RANDOM()), 2)
            ELSE ROUND(UNIFORM(-1.5, 2.0, RANDOM()), 2)
        END AS grid_import_export_kw
    FROM devices d
    CROSS JOIN hours h
    CROSS JOIN price_stats ps
    LEFT JOIN hourly_prices p ON p.price_hour = DATE_TRUNC('HOUR', h.ts);

    days_generated := :num_days;
    RETURN 'Generated telemetry for ' || :days_generated || ' days (' || :start_date || ' to ' || :end_date || ')';
END
$$


### Step 5c: Generate Telemetry Data

Execute the stored procedure to generate ~6.5M hourly telemetry records. The procedure auto-detects the time range: on first run it backfills 60 days; on subsequent runs it only fills gaps up to today. This is the heaviest data generation step in the setup — expect it to take 1–2 minutes on a Small warehouse.

In [ ]:
%%sql
-- Generate 60 days of telemetry using real day-ahead prices
CALL EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY();

-- Verify: should show ~60 days × ~4,500 battery devices × 24 hours ≈ 6.5M records
SELECT COUNT(*) AS telemetry_records,
       MIN(ts)::DATE AS earliest_date,
       MAX(ts)::DATE AS latest_date,
       DATEDIFF(DAY, MIN(ts), MAX(ts)) + 1 AS days_covered
FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY;

## 6. dbt Pipelines (Bronze → Silver → Gold)

The dbt project transforms raw data from Sections 4 and 5 through a **medallion architecture**. It runs **natively inside Snowflake** — no local dbt CLI, no external CI/CD, and no separate compute infrastructure required.

### What is dbt on Snowflake?

[dbt (data build tool)](https://docs.getdbt.com/) is a transformation framework that lets data engineers write modular SQL models with built-in testing, documentation, and dependency management. Snowflake has a **first-class dbt integration** that goes beyond the traditional dbt CLI:

| Capability | Traditional dbt | dbt on Snowflake |
|-----------|----------------|-----------------|
| **Execution** | Runs on a local machine or CI server | Runs inside Snowflake as a managed object (`EXECUTE DBT PROJECT`) |
| **Deployment** | Deployed via git + CI/CD pipelines | Deployed as a Snowflake object (`CREATE DBT PROJECT`) synced from a Git workspace |
| **Scheduling** | Requires external orchestrator (Airflow, cron, dbt Cloud) | Native Snowflake Tasks — fully serverless scheduling |
| **Compute** | Relies on external machine for orchestration | Uses Snowflake virtual warehouses — scale up/down on demand |
| **Governance** | Separate access control | Inherits Snowflake RBAC — roles, grants, and audit trails |
| **Monitoring** | External logging / dbt Cloud | Snowflake query history, task history, and dbt artifacts stored in Snowflake |

The key SQL commands for native dbt are:
- **`CREATE DBT PROJECT`** — deploys a dbt project from a Git workspace or stage into a Snowflake schema
- **`EXECUTE DBT PROJECT ... ARGS = 'run'`** — materializes all models (equivalent to `dbt run`)
- **`EXECUTE DBT PROJECT ... ARGS = 'test'`** — runs data quality assertions (equivalent to `dbt test`)

### Pipeline Overview

The pipeline is **one interconnected flow**, not two isolated streams. Both telemetry and price data converge in `mart_vpp_price_optimization` to answer the core business question: **Is the Virtual Power Plant profitable?**

```
 EPOWER_BRONZE (Raw)              EPOWER_SILVER (Cleaned)            EPOWER_GOLD (Business-Ready)
 ═══════════════════              ═════════════════════              ═════════════════════════════

 EPULSE_DEVICES ──────────► stg_devices (table) ◄─── customer_dim
                              │                      (EPOWER_GOLD)
                              │ (enriched with
                              │  customer context)
                              ▼
 RAW_EPULSE_IOT_ ────► fct_epulse_telemetry ──────► mart_vpp_capacity_hourly
 TELEMETRY               (table)                     (table: hourly fleet aggregation
                              │                       by region — solar, battery, grid)
                              │
                              │                      mart_vpp_price_optimization
                              │                      (table: THE JOIN — telemetry × prices)
                              └─────────────────────►  │ price_zone (NEGATIVE/LOW/MEDIUM/HIGH)
                                                       │ battery_action (CHARGE/DISCHARGE/...)
 RAW_DAY_AHEAD_ ──────► stg_day_ahead_prices ──────►  │ import_cost_eur (grid buy cost)
 PRICES                  (incremental)          │      │ export_revenue_eur (grid sell revenue)
                              │                 │      │ net_margin_eur (arbitrage profit)
                              │                 │      │ customer_margin_eur (70% share)
                              ▼                 │      │ epower_margin_eur (30% share)
                         mart_day_ahead_prices ─┘
                         (table: + EUR/kWh,
                          hour_of_day, day_of_week)
```

### Materialization Strategies

dbt supports different **materialization strategies** that control how models are stored in Snowflake:

| Strategy | Used For | Behavior |
|----------|---------|----------|
| **table** | `stg_devices`, `fct_epulse_telemetry`, all marts | Full rebuild on every run — simple and deterministic |
| **incremental** | `stg_day_ahead_prices` | Only processes new/changed rows — efficient for append-heavy data like daily price feeds |
| **view** | (not used here, but common for light transformations) | No storage cost — recomputed on every query |

In our `dbt_project.yml`, staging models use `table` materialization (for telemetry) or `incremental` (for prices), while all Gold marts are `table` to ensure fast analytical query performance.

### Goal: Quantify VPP Arbitrage Value

| Gold Model | Purpose | Grain |
|-----------|---------|-------|
| `mart_vpp_capacity_hourly` | **Fleet operations** — how much solar/battery capacity is active? | Per hour × region |
| `mart_vpp_price_optimization` | **Financial analysis** — arbitrage margins from buying cheap / selling expensive | Per hour × customer |
| `mart_day_ahead_prices` | **Market analytics** — price patterns, trends, volatility | Per 15-min interval |

The revenue model splits arbitrage margins **70% customer / 30% EPOWER**, following industry-standard VPP aggregator economics.

### Step 6a: Run & Test in Snowsight (Interactive)

Before deploying the dbt project as a Snowflake object, first run and test it interactively in the **Snowsight Workspace UI**:

1. Open the **Snowsight Workspace** (Projects → Workspaces → EPOWER_Demo)
2. Navigate to the `epower_dbt/` folder
3. Click **Run** to execute all models — watch the DAG visualization, model status, and logs
4. Click **Test** to validate data quality assertions
5. Inspect the Silver and Gold tables in the schema browser to verify the output

This interactive step helps you understand the pipeline before automating it.

### Step 6b: Deploy as Snowflake Object

Once validated in the UI, deploy the dbt project as a **managed Snowflake object** in `EPOWER_OPS` using `CREATE DBT PROJECT`. The project is synced from the Git-connected workspace — any future changes to the dbt models in the workspace can be re-deployed by re-running this statement. This makes the pipeline schedulable via Snowflake Tasks (Section 7).

The `CREATE DBT PROJECT` statement below deploys the dbt project from the Git-connected Snowsight workspace into a managed Snowflake object. The `FROM 'snow://workspace/...'` URI points to the live workspace files — Snowflake reads the `dbt_project.yml`, `profiles.yml`, model SQL files, and schema definitions directly from the workspace.

In [ ]:
%%sql -r dataframe_3
-- Step 6b: Deploy dbt project as a managed Snowflake object (synced via Git workspace)
CREATE OR REPLACE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT
    FROM 'snow://workspace/USER$.PUBLIC."EPOWER_Demo"/versions/live/epower_dbt';

### Step 6c: Scale Up Warehouse for Initial Pipeline Run

The first dbt run materializes all models from scratch — including `fct_epulse_telemetry` which processes ~9.7M rows and `mart_vpp_price_optimization` which joins telemetry with prices across millions of records. This is significantly more compute-intensive than the daily incremental runs that follow.

**Snowflake Elastic Scalability** — one of Snowflake's most powerful features is the ability to resize compute on demand, with no downtime and no data migration:

- **Vertical Scaling** (scale up/down) — increase warehouse size for more powerful single-query performance. A LARGE warehouse has 4× the compute resources of a SMALL, enabling complex joins and aggregations to complete faster. This is what we use here.
- **Horizontal Scaling** (scale out) — add multi-cluster capacity to handle more concurrent queries. Snowflake can automatically spin up additional clusters when query queuing is detected, distributing concurrent workloads across multiple identical warehouse clusters.

Both scaling dimensions take effect **within seconds** via a simple `ALTER WAREHOUSE` statement. After the pipeline completes, we scale back down to SMALL to minimize credit consumption — you only pay for what you use.

> **Gen2 Warehouses** — this demo uses a Generation 2 warehouse (`GENERATION = '2'`), which delivers improved performance through upgraded hardware and optimized software. Gen2 warehouses are particularly effective for DML-heavy workloads like the table materializations performed by dbt. See the [Snowflake Credit Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf) for Gen2 credit rates by size.

In [ ]:
%%sql
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'LARGE';
SELECT 'Warehouse scaled up to LARGE for initial dbt pipeline run' AS status;

### Step 6d: Run & Test the Pipeline

`EXECUTE DBT PROJECT ... ARGS = 'run'` materializes all models in dependency order — staging models first, then marts. This is equivalent to `dbt run` but executes entirely within Snowflake's compute infrastructure.

`EXECUTE DBT PROJECT ... ARGS = 'test'` then validates data quality: not-null constraints, uniqueness checks, and referential integrity tests defined in the dbt schema YAML files.

In [ ]:
%%sql
-- Run the deployed dbt project to build all Silver/Gold models
EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'run';

In [ ]:
%%sql
-- Test the deployed dbt project (data quality assertions)
EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT ARGS = 'test';

### Step 6e: Scale Down Warehouse

Pipeline complete — scale back to SMALL to reduce ongoing credit consumption. Daily incremental runs (via the scheduled Task in Section 7) process only new data and run efficiently on a Small warehouse.

In [ ]:
%%sql
ALTER WAREHOUSE EPOWER_COMPUTE SET WAREHOUSE_SIZE = 'SMALL';
SELECT 'Warehouse scaled back to SMALL' AS status;

## 7. Daily Task Scheduling

Now that the dbt project is deployed as a Snowflake object (`EPOWER_ANALYTICS_PROJECT`), it can be scheduled via a **Snowflake Task**.

The task runs daily at **17:00 CET** (day-ahead prices are published around 13:00 CET) and executes three steps in sequence:

1. **Fetch next-day prices** — `FETCH_DAY_AHEAD_PRICES(CURRENT_DATE() + 1)`
2. **Generate telemetry** — `GENERATE_DAILY_TELEMETRY()` fills gaps up to today
3. **Run dbt** — `EXECUTE DBT PROJECT` refreshes the full Bronze → Silver → Gold pipeline

In [ ]:
%%sql
CREATE OR REPLACE TASK EPOWER_DEMO.EPOWER_OPS.TASK_DAILY_DATA_REFRESH
    WAREHOUSE = EPOWER_COMPUTE
    SCHEDULE = 'USING CRON 0 17 * * * Europe/Berlin'
    COMMENT = 'Daily 17:00 CET: fetch next-day prices, generate VPP telemetry, run dbt (Bronze → Silver → Gold)'
AS
    BEGIN
        CALL EPOWER_DEMO.EPOWER_OPS.FETCH_DAY_AHEAD_PRICES(CURRENT_DATE() + 1);
        CALL EPOWER_DEMO.EPOWER_OPS.GENERATE_DAILY_TELEMETRY();
        EXECUTE DBT PROJECT EPOWER_DEMO.EPOWER_OPS.EPOWER_ANALYTICS_PROJECT 
            ARGS = 'run' 
            CONNECTION_NAME = 'default';
    END;

ALTER TASK EPOWER_DEMO.EPOWER_OPS.TASK_DAILY_DATA_REFRESH RESUME;
SHOW TASKS LIKE 'TASK_DAILY_DATA_REFRESH' IN SCHEMA EPOWER_DEMO.EPOWER_OPS;

## 8. Semantic Views for Cortex Analyst

Semantic Views enable natural language queries by defining business vocabulary, relationships, and metrics.

### Cortex Search Services for High-Cardinality Columns

Cortex Analyst needs exact literal values to generate correct SQL filters (e.g. `WHERE customer_name = 'Max Müller'`). For high-cardinality columns with many distinct values, we create **Cortex Search Services** that enable fuzzy semantic matching between the user's natural language input and the actual database values.

Each service indexes the distinct values of a column and is referenced in the semantic view dimensions via `WITH CORTEX SEARCH SERVICE`. When a user asks e.g. *"Show contracts for Mueller"*, the search service resolves this to the exact value `Max Müller` before SQL generation.

| Search Service | Column | Source Table | Semantic Views Using It |
|---|---|---|---|
| `_CA_CUSTOMER_NAME` | `customer_name` | `customer_dim` | Sales, Billing, Service, Customer Energy, VPP |
| `_CA_CITY` | `city` | `customer_dim` | Sales, VPP |
| `_CA_PRODUCT_NAME` | `product_name` | `product_dim` | Sales |
| `_CA_VENDOR_NAME` | `vendor_name` | `vendor_dim` | Sales |
| `_CA_REP_NAME` | `rep_name` | `sales_rep_dim` | Sales |

In [ ]:
%%sql
-- Cortex Search Services for High-Cardinality Columns (Dynamic Literal Retrieval)
CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME
  ON customer_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT customer_name FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY
  ON city
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT city FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_PRODUCT_NAME
  ON product_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT product_name FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_VENDOR_NAME
  ON vendor_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT vendor_name FROM EPOWER_DEMO.EPOWER_GOLD.VENDOR_DIM);

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_REP_NAME
  ON rep_name
  WAREHOUSE = EPOWER_COMPUTE
  TARGET_LAG = '1 hour'
  AS (SELECT DISTINCT rep_name FROM EPOWER_DEMO.EPOWER_GOLD.SALES_REP_DIM);

In [ ]:
%%sql
-- Energy Sales Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY) WITH SYNONYMS = ('Kunden','customers'),
        PRODUCTS AS EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM PRIMARY KEY (PRODUCT_KEY) WITH SYNONYMS = ('Produkte','products'),
        REGIONS AS EPOWER_DEMO.EPOWER_GOLD.REGION_DIM PRIMARY KEY (REGION_KEY) WITH SYNONYMS = ('Regionen','regions'),
        CONTRACTS AS EPOWER_DEMO.EPOWER_GOLD.SALES_FACT PRIMARY KEY (SALE_ID) WITH SYNONYMS = ('Vertraege','contracts'),
        SALES_REPS AS EPOWER_DEMO.EPOWER_GOLD.SALES_REP_DIM PRIMARY KEY (SALES_REP_KEY) WITH SYNONYMS = ('Berater','consultants'),
        VENDORS AS EPOWER_DEMO.EPOWER_GOLD.VENDOR_DIM PRIMARY KEY (VENDOR_KEY) WITH SYNONYMS = ('Lieferanten','vendors','partners')
    )
    RELATIONSHIPS (
        CONTRACTS_TO_CUSTOMERS AS CONTRACTS(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        CONTRACTS_TO_PRODUCTS AS CONTRACTS(PRODUCT_KEY) REFERENCES PRODUCTS(PRODUCT_KEY),
        CONTRACTS_TO_REGIONS AS CONTRACTS(REGION_KEY) REFERENCES REGIONS(REGION_KEY),
        CONTRACTS_TO_REPS AS CONTRACTS(SALES_REP_KEY) REFERENCES SALES_REPS(SALES_REP_KEY),
        CONTRACTS_TO_VENDORS AS CONTRACTS(VENDOR_KEY) REFERENCES VENDORS(VENDOR_KEY)
    )
    FACTS (
        CONTRACTS.CONTRACT_AMOUNT AS AMOUNT,
        CONTRACTS.CONTRACT_UNITS AS UNITS,
        PRODUCTS.CAPEX_EUR AS CAPEX_EUR,
        PRODUCTS.OPEX_EUR_YEAR AS OPEX_EUR_YEAR
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.CUSTOMER_TYPE AS CUSTOMER_TYPE,
        CUSTOMERS.CITY AS CITY WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY,
        PRODUCTS.PRODUCT_KEY AS PRODUCT_KEY,
        PRODUCTS.PRODUCT_NAME AS PRODUCT_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_PRODUCT_NAME,
        PRODUCTS.CATEGORY_NAME AS CATEGORY_NAME,
        REGIONS.REGION_KEY AS REGION_KEY,
        REGIONS.REGION_NAME AS REGION_NAME,
        CONTRACTS.SALE_ID AS SALE_ID,
        CONTRACTS.CONTRACT_DATE AS DATE,
        SALES_REPS.REP_NAME AS REP_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_REP_NAME,
        VENDORS.VENDOR_NAME AS VENDOR_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_VENDOR_NAME
    )
    METRICS (
        CONTRACTS.TOTAL_REVENUE AS SUM(CONTRACTS.CONTRACT_AMOUNT),
        CONTRACTS.TOTAL_CONTRACTS AS COUNT(CONTRACTS.SALE_ID),
        CONTRACTS.AVG_CONTRACT_VALUE AS AVG(CONTRACTS.CONTRACT_AMOUNT),
        PRODUCTS.AVG_CAPEX AS AVG(PRODUCTS.CAPEX_EUR),
        PRODUCTS.AVG_OPEX AS AVG(PRODUCTS.OPEX_EUR_YEAR)
    );

In [ ]:
%%sql
-- Billing Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        BILLING AS EPOWER_DEMO.EPOWER_GOLD.BILLING_HISTORY PRIMARY KEY (BILLING_ID) WITH SYNONYMS = ('Rechnungen','invoices')
    )
    RELATIONSHIPS (BILLING_TO_CUSTOMERS AS BILLING(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY))
    FACTS (
        BILLING.CONSUMPTION AS CONSUMPTION_KWH,
        BILLING.BILLING_AMOUNT AS AMOUNT
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.HOUSING_TYPE AS HOUSING_TYPE,
        BILLING.BILLING_ID AS BILLING_ID,
        BILLING.BILLING_DATE AS BILLING_DATE,
        BILLING.BILLING_TYPE AS BILLING_TYPE,
        BILLING.PAYMENT_STATUS AS PAYMENT_STATUS
    )
    METRICS (
        BILLING.TOTAL_CONSUMPTION AS SUM(BILLING.CONSUMPTION),
        BILLING.AVG_CONSUMPTION AS AVG(BILLING.CONSUMPTION)
    );

-- Service Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        TICKETS AS EPOWER_DEMO.EPOWER_GOLD.SERVICE_LOGS PRIMARY KEY (LOG_ID) WITH SYNONYMS = ('Tickets','Anfragen')
    )
    RELATIONSHIPS (TICKETS_TO_CUSTOMERS AS TICKETS(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY))
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        TICKETS.LOG_ID AS LOG_ID,
        TICKETS.LOG_DATE AS LOG_DATE,
        TICKETS.TOPIC AS TOPIC,
        TICKETS.CATEGORY AS CATEGORY,
        TICKETS.SENTIMENT AS SENTIMENT,
        TICKETS.PRIORITY AS PRIORITY
    )
    METRICS (
        TICKETS.TOTAL_TICKETS AS COUNT(TICKETS.LOG_ID)
    );

-- Customer Energy Semantic View (consumption + product ownership + VPP enrollment)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW
    TABLES (
        CUSTOMERS AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM PRIMARY KEY (CUSTOMER_KEY),
        BILLING AS EPOWER_DEMO.EPOWER_GOLD.BILLING_HISTORY PRIMARY KEY (BILLING_ID),
        OWNERSHIP AS EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_PRODUCTS PRIMARY KEY (CUSTOMER_PRODUCT_ID),
        DEVICES AS EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES PRIMARY KEY (CUSTOMER_KEY) WITH SYNONYMS = ('VPP','Virtual Power Plant','Geraete')
    )
    RELATIONSHIPS (
        BILLING_TO_CUSTOMERS AS BILLING(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        OWNERSHIP_TO_CUSTOMERS AS OWNERSHIP(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY),
        DEVICES_TO_CUSTOMERS AS DEVICES(CUSTOMER_KEY) REFERENCES CUSTOMERS(CUSTOMER_KEY)
    )
    FACTS (
        BILLING.CONSUMPTION AS CONSUMPTION_KWH,
        BILLING.BILLING_AMOUNT AS AMOUNT
    )
    DIMENSIONS (
        CUSTOMERS.CUSTOMER_KEY AS CUSTOMER_KEY,
        CUSTOMERS.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        CUSTOMERS.HOUSING_TYPE AS HOUSING_TYPE,
        BILLING.BILLING_DATE AS BILLING_DATE,
        BILLING.BILLING_TYPE AS BILLING_TYPE,
        OWNERSHIP.CATEGORY_NAME AS CATEGORY_NAME,
        DEVICES.GATEWAY_ID AS GATEWAY_ID,
        DEVICES.HAS_SOLAR AS HAS_SOLAR,
        DEVICES.HAS_BATTERY AS HAS_BATTERY,
        DEVICES.HAS_HEATPUMP AS HAS_HEATPUMP,
        DEVICES.IS_VPP_ENROLLED AS IS_VPP_ENROLLED,
        DEVICES.ENROLLMENT_DATE AS ENROLLMENT_DATE
    )
    METRICS (
        BILLING.AVG_CONSUMPTION AS AVG(BILLING.CONSUMPTION)
    );

-- HR Semantic View
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW
    TABLES (
        DEPARTMENTS AS EPOWER_DEMO.EPOWER_GOLD.DEPARTMENT_DIM PRIMARY KEY (DEPARTMENT_KEY),
        EMPLOYEES AS EPOWER_DEMO.EPOWER_GOLD.EMPLOYEE_DIM PRIMARY KEY (EMPLOYEE_KEY),
        HR_RECORDS AS EPOWER_DEMO.EPOWER_GOLD.HR_EMPLOYEE_FACT PRIMARY KEY (HR_FACT_ID),
        JOBS AS EPOWER_DEMO.EPOWER_GOLD.JOB_DIM PRIMARY KEY (JOB_KEY)
    )
    RELATIONSHIPS (
        HR_TO_DEPARTMENTS AS HR_RECORDS(DEPARTMENT_KEY) REFERENCES DEPARTMENTS(DEPARTMENT_KEY),
        HR_TO_EMPLOYEES AS HR_RECORDS(EMPLOYEE_KEY) REFERENCES EMPLOYEES(EMPLOYEE_KEY),
        HR_TO_JOBS AS HR_RECORDS(JOB_KEY) REFERENCES JOBS(JOB_KEY)
    )
    FACTS (
        HR_RECORDS.ATTRITION_FLAG AS ATTRITION_FLAG,
        HR_RECORDS.EMPLOYEE_SALARY AS SALARY
    )
    DIMENSIONS (
        DEPARTMENTS.DEPARTMENT_NAME AS DEPARTMENT_NAME,
        EMPLOYEES.EMPLOYEE_NAME AS EMPLOYEE_NAME,
        EMPLOYEES.GENDER AS GENDER,
        JOBS.JOB_TITLE AS JOB_TITLE
    )
    METRICS (
        HR_RECORDS.TOTAL_SALARY AS SUM(HR_RECORDS.EMPLOYEE_SALARY),
        HR_RECORDS.AVG_SALARY AS AVG(HR_RECORDS.EMPLOYEE_SALARY)
    );

In [ ]:
%%sql
-- ePulse Day-Ahead Prices Semantic View (Energy Market)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW
    TABLES (
        PRICES AS EPOWER_DEMO.EPOWER_SILVER.STG_DAY_AHEAD_PRICES PRIMARY KEY (START_TIME)
            WITH SYNONYMS = ('Strompreise','electricity prices','day-ahead','Boersenpreise','spot prices','Marktpreise','energy market prices')
    )
    FACTS (
        PRICES.PRICE_EUR_MWH AS PRICE_EUR_MWH
    )
    DIMENSIONS (
        PRICES.START_TIME AS START_TIME,
        PRICES.END_TIME AS END_TIME,
        PRICES.FETCH_DATE AS FETCH_DATE
    )
    METRICS (
        PRICES.AVG_PRICE AS AVG(PRICES.PRICE_EUR_MWH),
        PRICES.MAX_PRICE AS MAX(PRICES.PRICE_EUR_MWH),
        PRICES.MIN_PRICE AS MIN(PRICES.PRICE_EUR_MWH)
    );

In [ ]:
%%sql
-- ePulse VPP Telemetry Semantic View (Virtual Power Plant IoT)
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW
    TABLES (
        TELEMETRY AS EPOWER_DEMO.EPOWER_SILVER.FCT_EPULSE_TELEMETRY PRIMARY KEY (TS)
            WITH SYNONYMS = ('VPP Telemetrie','virtual power plant','ePulse','IoT','Batteriespeicher','battery storage','solar','grid')
    )
    FACTS (
        TELEMETRY.SOLAR_YIELD_KW AS SOLAR_YIELD_KW,
        TELEMETRY.BATTERY_SOC_PCT AS BATTERY_SOC_PCT,
        TELEMETRY.HEATPUMP_CONSUMPTION_KW AS HEATPUMP_CONSUMPTION_KW,
        TELEMETRY.GRID_IMPORT_EXPORT_KW AS GRID_IMPORT_EXPORT_KW
    )
    DIMENSIONS (
        TELEMETRY.TS AS TS,
        TELEMETRY.GATEWAY_ID AS GATEWAY_ID,
        TELEMETRY.CUSTOMER_KEY AS CUSTOMER_KEY,
        TELEMETRY.CUSTOMER_NAME AS CUSTOMER_NAME WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CUSTOMER_NAME,
        TELEMETRY.CITY AS CITY WITH CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD._CA_CITY,
        TELEMETRY.REGION AS REGION,
        TELEMETRY.CUSTOMER_TYPE AS CUSTOMER_TYPE,
        TELEMETRY.IS_VPP_ENROLLED AS IS_VPP_ENROLLED
    )
    METRICS (
        TELEMETRY.AVG_SOLAR_YIELD AS AVG(TELEMETRY.SOLAR_YIELD_KW),
        TELEMETRY.TOTAL_SOLAR_YIELD AS SUM(TELEMETRY.SOLAR_YIELD_KW),
        TELEMETRY.AVG_BATTERY_SOC AS AVG(TELEMETRY.BATTERY_SOC_PCT),
        TELEMETRY.AVG_HEATPUMP_CONSUMPTION AS AVG(TELEMETRY.HEATPUMP_CONSUMPTION_KW),
        TELEMETRY.NET_GRID_FLOW AS SUM(TELEMETRY.GRID_IMPORT_EXPORT_KW),
        TELEMETRY.READING_COUNT AS COUNT(TELEMETRY.TS)
    );

## 9. Document Parsing & Cortex Search

In [ ]:
%%sql
-- Create stage with server-side encryption for document parsing
CREATE OR REPLACE STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
    DIRECTORY = (ENABLE = TRUE);

-- Copy unstructured documents to the encrypted stage
COPY FILES INTO @EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE/unstructured_data/ FROM @EPOWER_DEMO.EPOWER_OPS.EPOWER_STAGE/unstructured_data/;
ALTER STAGE EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE REFRESH;

-- Parse PDF/MD documents using AI_PARSE_DOCUMENT
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT AS 
SELECT 
    relative_path, 
    BUILD_STAGE_FILE_URL('@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE', relative_path) as file_url,
    AI_PARSE_DOCUMENT(TO_FILE('@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE', relative_path), {'mode':'LAYOUT'}):content::string as content
FROM directory(@EPOWER_DEMO.EPOWER_OPS.EPOWER_DOCS_STAGE) 
WHERE relative_path ILIKE 'unstructured_data/%.pdf' OR relative_path ILIKE 'unstructured_data/%.md';

SELECT COUNT(*) AS parsed_docs FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT;

In [ ]:
%%sql
-- Create Cortex Search services for RAG
CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/energy/%');

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/products/%');

CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS ON content ATTRIBUTES relative_path, file_url
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '30 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT relative_path, file_url, content FROM EPOWER_DEMO.EPOWER_GOLD.PARSED_CONTENT WHERE relative_path ILIKE '%/service/%');

 CREATE OR REPLACE CORTEX SEARCH SERVICE EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS ON description ATTRIBUTES log_id, customer_key, topic, sentiment
    WAREHOUSE = EPOWER_COMPUTE TARGET_LAG = '1 day' EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (SELECT log_id, customer_key, topic, description, sentiment FROM EPOWER_DEMO.EPOWER_GOLD.SERVICE_LOGS);

## 10. Snowflake Intelligence Agent

In [ ]:
%%sql
-- Network rule for web access
CREATE OR REPLACE NETWORK RULE EPOWER_DEMO.EPOWER_OPS.ENERGY_WEBACCESSRULE MODE = EGRESS TYPE = HOST_PORT VALUE_LIST = ('0.0.0.0:443');

USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION ENERGY_EXTERNALACCESS ALLOWED_NETWORK_RULES = (EPOWER_DEMO.EPOWER_OPS.ENERGY_WEBACCESSRULE) ENABLED = true;
GRANT USAGE ON INTEGRATION ENERGY_EXTERNALACCESS TO ROLE EPOWER_ROLE;
USE ROLE EPOWER_ROLE;

In [ ]:
%%sql -r dataframe_2
-- Create the Intelligence Agent
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question. If the user asks in English, respond entirely in English. If the user asks in German, respond entirely in German. Never switch languages unless the user does. Match the language of every question independently.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export, IoT → vpp_telemetry_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices, spot prices, energy market"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery state-of-charge, heatpump consumption, grid import/export per device and region"}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

In [ ]:
%%sql
USE ROLE accountadmin;
CREATE SNOWFLAKE INTELLIGENCE IF NOT EXISTS snowflake_intelligence_object_default;
ALTER SNOWFLAKE INTELLIGENCE snowflake_intelligence_object_default ADD AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT;
GRANT USAGE ON AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT TO ROLE PUBLIC;
USE ROLE EPOWER_ROLE;

## 11. MCP Server (Model Context Protocol)

In [ ]:
%%sql
CREATE OR REPLACE MCP SERVER EPOWER_DEMO.EPOWER_GOLD.EPOWER_MCP_SERVER
FROM SPECIFICATION $$
tools:
  - name: "epower-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT"
    description: "EPOWER Energy Intelligence Agent — answers questions across sales, billing, service, HR, VPP telemetry, market prices, and energy documents using text-to-SQL and RAG."
    title: "EPOWER Intelligence Agent"

  - name: "energy-sales-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"
    description: "Contracts, products, sales revenue, and customer data"
    title: "Energy Sales Analyst"

  - name: "billing-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"
    description: "Customer consumption, billing history, and payments"
    title: "Billing Analyst"

  - name: "service-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"
    description: "Service tickets, complaints, and resolution data"
    title: "Service Analyst"

  - name: "customer-energy-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"
    description: "Consumption analysis by product ownership (heat pumps, solar, etc.)"
    title: "Customer Energy Analyst"

  - name: "hr-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"
    description: "HR workforce data, salaries, departments"
    title: "HR Analyst"

  - name: "market-prices-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"
    description: "Day-ahead electricity spot prices from Energy-Charts"
    title: "Market Prices Analyst"

  - name: "vpp-telemetry-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"
    description: "VPP IoT telemetry: solar yield, battery SOC, heat pump consumption, grid import/export"
    title: "VPP Telemetry Analyst"

  - name: "energy-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS"
    description: "Search energy policies, terms and conditions"
    title: "Energy Documents Search"

  - name: "product-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS"
    description: "Search product documentation and specifications"
    title: "Product Documents Search"

  - name: "service-docs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS"
    description: "Search service handbook and procedures"
    title: "Service Documents Search"

  - name: "service-logs-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS"
    description: "Search historical service ticket logs"
    title: "Service Logs Search"
$$;

## 12. Verification

In [ ]:
%%sql -r dataframe_1
SELECT * FROM EPOWER_DEMO.INFORMATION_SCHEMA.SEMANTIC_VIEWS;

In [ ]:
%%sql
-- Summary of all created objects
SELECT 'Gold Tables' AS category, COUNT(*) AS count FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_GOLD' AND TABLE_TYPE='BASE TABLE'
UNION ALL SELECT 'Bronze Tables', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_BRONZE'
UNION ALL SELECT 'Ops Objects', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_OPS'
UNION ALL SELECT 'Semantic Views', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.SEMANTIC_VIEWS WHERE SCHEMA='EPOWER_GOLD' AND NAME LIKE '%SEMANTIC%'
UNION ALL SELECT 'Cortex Search', COUNT(*) FROM EPOWER_DEMO.INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA='EPOWER_GOLD' AND TABLE_NAME LIKE 'SEARCH_%';

In [ ]:
%%sql
-- Verify ePulse VPP data
SELECT 'EPULSE_DEVICES' AS table_name, COUNT(*) AS row_count FROM EPOWER_DEMO.EPOWER_BRONZE.EPULSE_DEVICES
UNION ALL SELECT 'RAW_EPULSE_IOT_TELEMETRY', COUNT(*) FROM EPOWER_DEMO.EPOWER_BRONZE.RAW_EPULSE_IOT_TELEMETRY
UNION ALL SELECT 'EPOWER_SILVER.STG_DEVICES', COUNT(*) FROM EPOWER_DEMO.EPOWER_SILVER.STG_DEVICES
UNION ALL SELECT 'EPOWER_GOLD.MART_VPP_CAPACITY_HOURLY', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.MART_VPP_CAPACITY_HOURLY;

## Setup Complete!

**What was created:**
- **20K customers** with 15 products (CAPEX/OPEX pricing) across 6 energy categories
- **ePulse VPP** with 60 days of **price-reactive** IoT telemetry correlated with real EPEX day-ahead prices
- **60 days of real day-ahead prices** from Energy-Charts API (CET delivery-day aligned)
- **7 Semantic Views** for natural language queries via Cortex Analyst
- **4 Cortex Search** services for document RAG
- **5 Cortex Search** services for high-cardinality column lookup
- **1 Intelligence Agent** combining all capabilities
- **1 Scheduled Task** for daily data refresh (prices + telemetry + dbt)
- **1 MCP Server** exposing Agent, Semantic Views, and Search Services via Model Context Protocol

**Semantic Views → Agent Tools:**
| Semantic View | Agent Tool | Domain |
|---|---|---|
| `ENERGY_SALES_SEMANTIC_VIEW` | `energy_sales_analyst` | Contracts, products, sales, revenue |
| `BILLING_SEMANTIC_VIEW` | `billing_analyst` | Consumption, billing, payments |
| `CUSTOMER_ENERGY_SEMANTIC_VIEW` | `customer_energy_analyst` | Consumption by product ownership |
| `SERVICE_SEMANTIC_VIEW` | `service_analyst` | Service tickets, complaints |
| `HR_SEMANTIC_VIEW` | `hr_analyst` | HR data, salaries |
| `MARKET_PRICES_SEMANTIC_VIEW` | `epulse_prices_analyst` | Day-ahead electricity market prices |
| `EPULSE_VPP_SEMANTIC_VIEW` | `vpp_telemetry_analyst` | VPP IoT: solar, battery, heatpump, grid |

**Try the Agent:**
- "What were total sales by region last month?"
- "Zeige mir alle negativen Service-Tickets zum Thema Smart Meter"
- "What's the average consumption for customers with heat pumps?"
- "What are the current day-ahead electricity prices?"
- "Show me the average battery state-of-charge for VPP-enrolled customers"
- "Summarize our Green Power policy" 